In [5]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")


In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)


In [7]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [8]:
q = ground_truth[0]
q


{'question': 'I just found out about this course, is it too late to start learning with you guys?',
 'course': 'llm-zoomcamp',
 'document': '74eb249bbf'}

In [9]:
doc_id = q["document"]
results = text_search(query=q["question"])


In [10]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance


[0, 0, 0, 0, 0]

In [11]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance


In [12]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]


I just found out about this course, is it too late to start learning with you guys?


[0, 0, 0, 0, 0]

In [13]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]


Do students get the zoom link for the llm zoomcamp office hours? I can't find it anywhere.


[1, 0, 0, 0, 0]

In [14]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]


Where can I find the LLM Zoomcamp syllabus and deadlines? I keep losing track of what's due when.


[1, 0, 0, 0, 0]

In [15]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total


In [16]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)


  0%|          | 0/15 [00:00<?, ?it/s]

In [17]:
relevance_total_text


[[0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [18]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance


In [19]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total


In [20]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total


  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [21]:
relevance_total = compute_relevance_total(ground_truth, text_search)


  0%|          | 0/804 [00:00<?, ?it/s]

### Search Evaluation Metrics

In [28]:
# Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere


example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]


In [29]:
cnt = 0

for line in example:
    if 1 in line:
        cnt += 1
        
cnt

14

In [30]:
cnt / len(example)

0.9333333333333333

In [31]:
def hit_rate(relevance):
    cnt = 0
    
    for line in relevance:
        if 1 in line:
            cnt += 1
    return cnt / len(relevance)

In [32]:
hit_rate(example)

0.9333333333333333

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

position 1: score is 1.0
position 2: score is 0.5
position 3: score is 0.333
not found: score is 0
In the example, most hits are at the first position. Some hits are lower in the list.



In [33]:
example[1]
# [0, 1, 0, 0, 0]


[0, 1, 0, 0, 0]

In [34]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score += 1 / (rank + 1)
            break
        
total_score

12.333333333333332

In [36]:
total_score / len(example)
# 0.822


0.8222222222222222

In [37]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


In [38]:
mrr(example)
# 0.822


0.8222222222222222

In [39]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }


In [40]:
evaluate(
    ground_truth,
    text_search
)


  0%|          | 0/804 [00:00<?, ?it/s]

{'hit_rate': 0.835820895522388, 'mrr': 0.7459577114427858}

### Search Parameter Tuning

In [41]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )


In [42]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")


  0%|          | 0/804 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.835820895522388, 'mrr': 0.7459577114427858}


  0%|          | 0/804 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.835820895522388, 'mrr': 0.7459577114427858}


  0%|          | 0/804 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.835820895522388, 'mrr': 0.7459577114427858}


  0%|          | 0/804 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.835820895522388, 'mrr': 0.7459577114427858}


  0%|          | 0/804 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.835820895522388, 'mrr': 0.7459577114427858}


In [43]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )


In [ ]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/804 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/804 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


  0%|          | 0/804 [00:00<?, ?it/s]

... (46 outputs left out)

Evaluating question_boost=5.0, answer_boost=1.0, section_boost=0.5...


  0%|          | 0/804 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=2.0, section_boost=0.1...


  0%|          | 0/804 [00:00<?, ?it/s]

In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)


In [ ]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )
